# De una sola imagen a la fracción de franja: el método de Fourier

**Objetivo.** Recorrer, de la teoría a la ejecución, el método de Fourier de una sola toma tal como lo implementa
`interflib.fringe.fourier`: cómo convierte **un único interferograma** con franjas portadoras en un mapa de **fase
envuelta** y, con él, en la **fracción de franja** $F$ de un bloque patrón adherido a su platina. Las etapas son:

1. **Portadora:** estimar la frecuencia de las franjas a lo largo de las columnas.
2. **Demodulación:** aislar en el espectro el lóbulo de la franja y transformarlo de vuelta; su ángulo es la fase.
3. **Fase de cada banda:** desenvolverla a lo largo de la columna y leerla en la fila de evaluación.
4. **Fracción:** extrapolar la fase de platina y bloque hasta el borde del bloque y restarlas allí.

Las secciones 3 a 7 recorren la versión por columnas; la sección 8, la variante bidimensional, que demodula de una
vez una región de cada superficie.

Cada bloque de código **importa y ejecuta las funciones reales de `interflib`**; no reimplementa nada. El banco de
pruebas es el simulador por capas `interflib.simulation`, que entrega la **verdad de terreno** (la fase y la fracción
$F=\operatorname{frac}(2L/\lambda)$) para comprobar cada paso. El cuaderno muestra el **funcionamiento** del método;
no evalúa su desempeño en condiciones de medición.

## 0. Preparación

Se importan las funciones públicas del método y del simulador. Convenciones de la librería: las imágenes son filas
$\times$ columnas, con $x$ las columnas e $y$ las filas; las franjas son aproximadamente **horizontales**, de modo que
la portadora corre a lo largo de $y$; la platina queda a la izquierda del borde del bloque y el bloque a la derecha.
Símbolos: $\varphi$ fase envuelta, $a$ fondo (nuestro $I_0$), $b$ amplitud de modulación (nuestro $I_0V$), $g_y=1/\Lambda$
portadora (franjas por píxel, con $\Lambda$ el periodo), $F$ fracción de franja.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from interflib.simulation import (
    SceneGeometry, PhaseModel, Illumination, SensorModel, synthesize_interferogram,
)
from interflib.fringe.fourier import (
    estimate_carrier, demodulate_columns, fourier_column_phases, fourier_edge_fraction,
    estimate_carrier_2d, demodulate_2d, fourier_edge_fraction_2d,
)
from interflib.fringe.unwrap import wrap_phase

plt.rcParams.update({"figure.dpi": 110, "image.cmap": "gray", "font.size": 10})
np.set_printoptions(precision=4, suppress=True)

# --- Escena: bloque de 10 mm sobre su platina, franjas horizontales -------------------
WAVELENGTH_NM = 632.8        # He-Ne rojo
CENTRAL_LENGTH_NM = 1.0e7    # bloque patron de 10 mm
geometry = SceneGeometry(resolution=(480, 640), fov_mm=25.0,
                         block_width_mm=9.0, block_height_mm=10.0)
phase_model = PhaseModel(wavelength_nm=WAVELENGTH_NM, central_length_nm=CENTRAL_LENGTH_NM,
                         fringes_x=1.5, fringes_y=24.0)      # leve inclinacion + portadora en y
illumination = Illumination(background=0.5, vignette=0.12, visibility=0.85,
                            visibility_falloff=0.15, psf_sigma_px=0.0)
clean_sensor = SensorModel(shot_noise=False, read_noise_e=0.0, bit_depth=16, saturate=False)

scene = synthesize_interferogram(geometry, phase_model, illumination, clean_sensor, seed=0)
img = scene.irradiance

# Borde izquierdo del bloque (platina a su izquierda) y filas del bloque
cols = np.where(scene.block_mask.any(axis=0))[0]
rows = np.where(scene.block_mask.any(axis=1))[0]
EDGE_X = cols[0] - 0.5
EVAL_ROW = int(round(rows.mean()))
FFT_ROWS = (int(rows[0]) + 28, int(rows[-1]) - 27)   # intervalo de la transformada, dentro del bloque
ROWS = (EVAL_ROW - 60, EVAL_ROW + 60)          # filas cuya fase entra en el ajuste


def show(im, title="", cmap="gray", ax=None, colorbar=False, **kw):
    if ax is None:
        _, ax = plt.subplots(figsize=(4, 3))
    h = ax.imshow(im, cmap=cmap, **kw)
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
    if colorbar:
        plt.colorbar(h, ax=ax, fraction=0.046, pad=0.04)
    return ax


print("imagen:", img.shape, "| borde del bloque en x =", EDGE_X, "| bloque en filas", rows[0], "a", rows[-1])
print("fila de evaluacion:", EVAL_ROW, "| transformada en filas", FFT_ROWS, "| ventana", ROWS)
print("fraccion verdadera F = frac(2L/lambda) =", round(scene.fraction_truth, 6))

## 1. El problema: de la intensidad a la fase, con una sola toma

Un interferograma de dos haces obedece, píxel a píxel,

$$I(x,y) = a(x,y) + b(x,y)\,\cos\varphi(x,y).$$

Recuperar $\varphi$ de una sola imagen es un problema mal planteado: tres incógnitas por píxel y una ecuación, con la
ambigüedad de signo del coseno y la de $2\pi$ [Servín, Quiroga & Padilla, *Fringe Pattern Analysis for Optical
Metrology* (2014), §1.1]. El desplazamiento de fase resuelve el problema con una portadora **temporal** (varias
tomas). El método de Fourier usa una portadora **espacial**: con el espejo de referencia inclinado, la fase crece de
forma conocida a lo largo de la imagen,

$$\varphi(x,y) = 2\pi g_y\,y + \phi(x,y),$$

donde $g_y=1/\Lambda$ es la portadora y $\phi$ lo que se quiere medir. Escribiendo el coseno como suma de
exponenciales,

$$I = a + \tfrac{b}{2}\,e^{i\phi}e^{\,i2\pi g_y y} + \tfrac{b}{2}\,e^{-i\phi}e^{-i2\pi g_y y},$$

el espectro de $I$ a lo largo de $y$ tiene **tres lóbulos**: el fondo $a$ cerca de la frecuencia cero y dos copias de
la franja en $\pm g_y$. Si $a$, $b$ y $\phi$ varían despacio frente a la franja, los lóbulos no se solapan. Basta
entonces quedarse con el lóbulo positivo y transformar de vuelta para obtener la **señal analítica**
$\tfrac{b}{2}e^{i\varphi}$, cuyo ángulo es la fase y cuyo módulo es la mitad de la modulación [Takeda, Ina &
Kobayashi, *J. Opt. Soc. Am.* **72**, 156 (1982); Servín *et al.* (2014), §4.2].

**Por qué por columnas.** La librería hace la transformada **a lo largo de cada columna**, no sobre la imagen 2D. Con
franjas casi horizontales y el borde del bloque vertical, una columna es entera de bloque o entera de platina, de modo
que el escalón del borde no entra en el espectro de ninguna columna. Una inclinación de las franjas solo añade un
gradiente de fase de una columna a otra, que el método sigue sin más.

## 2. La escena

El simulador genera un bloque de 10 mm adherido a su platina, con unas 24 franjas horizontales a lo largo de la imagen
y una leve inclinación, sin ruido de cámara para ver el método en limpio.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
show(img, title="interferograma", ax=axes[0])
axes[0].axvline(EDGE_X, color="tab:red", lw=1)
axes[0].axvspan(EDGE_X - 180, EDGE_X - 30, color="tab:blue", alpha=0.15)
axes[0].axvspan(EDGE_X + 30, EDGE_X + 180, color="tab:orange", alpha=0.15)
axes[0].axhspan(*FFT_ROWS, fill=False, ec="tab:green", ls="--", lw=1.2)
axes[0].axhspan(*ROWS, fill=False, ec="tab:purple", lw=1.2)
show(scene.block_mask, title="mascara del bloque (verdad)", ax=axes[1], cmap="gray")
plt.tight_layout(); plt.show()

**Lectura de la figura.** A la izquierda, el interferograma: el bloque es el rectángulo central, cuyas franjas están
corridas respecto de las de la platina que lo rodea; ese corrimiento, medido en fracción de franja, es $F$. La línea
roja es el borde izquierdo del bloque; las bandas azul y naranja, las columnas que se usarán a cada lado (de 30 a
180 px del borde, para dejar fuera la zona donde el escalón difracta); el recuadro verde, las filas de la
transformada, y el morado, la ventana de filas cuya fase entra en el ajuste, centrada en la fila de evaluación. A la
derecha, la máscara del bloque que entrega el simulador: el bloque tiene bordes **horizontales** dentro del campo
(arriba y abajo), y el intervalo de la transformada queda entre ellos (sección 5).

## 3. La portadora

`estimate_carrier` promedia el espectro de potencia de varias bandas de columnas y toma su **pico global** (sin la
frecuencia cero), refinado entre muestras con una parábola. Si ese pico cae por debajo de un mínimo de franjas en el
intervalo (`min_cycles`, 4 por defecto), el lóbulo no se separa del fondo y la función lo rechaza en lugar de
devolver un valor sin sentido.

In [ ]:
x_band = int(EDGE_X - 100)                       # una banda de la platina
profile = img[FFT_ROWS[0]:FFT_ROWS[1], x_band - 2:x_band + 3].mean(axis=1)
g_y = estimate_carrier(profile[:, None])

n = len(profile)
freqs = np.fft.fftshift(np.fft.fftfreq(n))
spectrum = np.fft.fftshift(np.abs(np.fft.fft(profile)))
half = 0.5 * g_y
passband = np.where(np.abs(freqs - g_y) < half, np.cos(np.pi * (freqs - g_y) / (2 * half)) ** 2, 0.0)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
axes[0].plot(np.arange(*FFT_ROWS), profile, lw=1)
axes[0].set_xlabel("fila y (px)"); axes[0].set_ylabel("intensidad"); axes[0].set_title("perfil de una banda de la platina")
axes[1].semilogy(freqs, spectrum + 1e-12, "k", lw=1, label="|espectro|")
ax2 = axes[1].twinx()
ax2.fill_between(freqs, passband, color="tab:green", alpha=0.25, label="filtro")
ax2.set_ylim(0, 1.5); ax2.set_yticks([])
axes[1].axvline(g_y, color="tab:green", ls=":"); axes[1].axvline(-g_y, color="tab:gray", ls=":")
axes[1].set_xlim(-0.15, 0.15); axes[1].set_xlabel("frecuencia (franjas/px)")
axes[1].set_title("espectro y filtro pasabanda")
plt.tight_layout(); plt.show()
print(f"portadora g_y = {g_y:.5f} franjas/px  ->  periodo {1 / g_y:.2f} px, "
      f"{g_y * n:.1f} franjas en el intervalo de {n} filas")

**Lectura de la figura.** A la izquierda, el perfil de intensidad de una banda de cinco columnas de la platina a lo
largo de las filas de la transformada: un coseno sobre un fondo que varía despacio. A la derecha, su espectro en
escala logarítmica: el pico central es el fondo $a$ y los dos laterales, en $\pm g_y$, son la franja. El área verde es
el filtro que usará la demodulación: un coseno alzado centrado en $g_y$ con semiancho $0{,}5\,g_y$ (parámetro
`bandwidth`), que deja pasar solo el lóbulo positivo y deja fuera el fondo. Cuanto mayor es $g_y$ (más franjas),
más separados están los lóbulos del fondo.

## 4. Demodulación: fase, modulación y fondo en cada píxel

`demodulate_columns` aplica, banda por banda, la transformada a lo largo de $y$, el filtro y la transformada inversa.
El ángulo de la señal analítica es la fase envuelta; el doble de su módulo, la modulación $b$; un pasabajos de la misma
anchura da el fondo $a$. La fase se devuelve en la convención del ajuste sinusoidal de la librería (la fase $\phi$ de
$A\sin(\omega y+\phi)+C$), es decir, desplazada $\pi/2$ respecto del ángulo del coseno, para que los dos métodos den
fases comparables.

In [ ]:
xs = np.arange(10, img.shape[1] - 10, 2)
dem = demodulate_columns(img, xs, FFT_ROWS)
y = np.arange(*FFT_ROWS)

# Error frente a la fase verdadera (sin el desfase constante de pi/2 de la convencion)
err = wrap_phase(dem.phase - scene.phase[FFT_ROWS[0]:FFT_ROWS[1]][:, xs])
err = wrap_phase(err - np.angle(np.mean(np.exp(1j * err))))

fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
ext = [xs[0], xs[-1], FFT_ROWS[1], FFT_ROWS[0]]
for ax, data, title, cmap in [(axes[0], dem.phase, r"fase envuelta $\varphi$", "twilight"),
                              (axes[1], dem.modulation, r"modulacion $b$", "viridis"),
                              (axes[2], dem.background, r"fondo $a$", "viridis"),
                              (axes[3], err, r"$\varphi$ - verdad (rad)", "RdBu_r")]:
    kw = dict(vmin=-0.01, vmax=0.01) if ax is axes[3] else {}
    h = ax.imshow(data, cmap=cmap, aspect="auto", extent=ext, **kw)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
    plt.colorbar(h, ax=ax, fraction=0.046, pad=0.04)
    ax.axvline(EDGE_X, color="k", lw=0.5); ax.axvline(cols[-1] + 0.5, color="k", lw=0.5)
plt.tight_layout(); plt.show()

inner = (y >= ROWS[0]) & (y < ROWS[1])
far = (np.abs(xs - EDGE_X) > 30) & (np.abs(xs - (cols[-1] + 0.5)) > 30)
print(f"|fase - verdad| en la ventana, lejos de los bordes: max {np.abs(err[inner][:, far]).max():.1e} rad")

**Lectura de la figura.** De izquierda a derecha: la fase envuelta, que salta $2\pi$ en cada franja y muestra el
corrimiento del bloque entre las dos líneas negras (sus bordes laterales); la modulación $b$ y el fondo $a$, que
reproducen el haz del simulador; y la diferencia entre la fase demodulada y la verdadera, en una escala de
$\pm 0{,}01$ rad. Esa diferencia es casi nula en el interior y crece solo en los **extremos del intervalo** de la
transformada, porque el espectro de una señal cortada se reparte (fuga espectral). Las columnas junto a los bordes
laterales del bloque no muestran error: cada columna es entera de bloque o de platina y el borde no entra en su
espectro. En una imagen real el borde sí se ve como un valle oscuro de unos píxeles, por la difracción del escalón; por
eso las bandas empiezan a 30 px del borde. La ventana de la fase se toma bien dentro del intervalo.

## 5. El intervalo de la transformada no debe cruzar un borde horizontal del bloque

Las columnas del bloque se cortan arriba y abajo, donde termina el bloque. Si el intervalo de la transformada cruza
ese corte, la columna deja de ser una franja continua: el salto de fase y de intensidad se reparte por el espectro y
contamina la fase en una franja de filas alrededor del corte, de un ancho del orden de $1/(\mathrm{bandwidth}\cdot
g_y)$ píxeles. Se compara la demodulación de una columna del bloque con el intervalo dentro del bloque y con un
intervalo que cruza sus dos bordes.

In [ ]:
x_block = int(EDGE_X + 115)
wide = (20, img.shape[0] - 20)
fig, ax = plt.subplots(figsize=(10, 3.4))
for span, c, lab in [(wide, "tab:red", "intervalo que cruza los bordes del bloque"),
                     (FFT_ROWS, "tab:green", "intervalo dentro del bloque")]:
    d = demodulate_columns(img, [x_block], span)
    yy = np.arange(*span)
    e = wrap_phase(d.phase[:, 0] - scene.phase[span[0]:span[1], x_block])
    e = wrap_phase(e - np.angle(np.mean(np.exp(1j * e[(yy >= ROWS[0]) & (yy < ROWS[1])]))))
    ax.plot(yy, e, color=c, lw=1, label=lab)
for r in (rows[0], rows[-1]):
    ax.axvline(r, color="k", ls=":", lw=1)
ax.axvspan(*ROWS, color="tab:purple", alpha=0.08, label="ventana de la fase")
ax.set_ylim(-0.3, 0.3); ax.set_xlabel("fila y (px)"); ax.set_ylabel("fase - verdad (rad)")
ax.set_title(f"columna {x_block} del bloque", fontsize=10); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

**Lectura de la figura.** Error de la fase demodulada a lo largo de una columna del bloque; las líneas de puntos son
los bordes superior e inferior del bloque. Con el intervalo que los cruza (rojo), el error se dispara en las filas
vecinas a cada borde y se extiende varias decenas de filas hacia dentro. Con el intervalo dentro del bloque (verde),
el error solo crece en los extremos del propio intervalo. En los dos casos la ventana de la fase (sombreada) queda
lejos de los extremos, pero el intervalo que cruza los bordes desperdicia filas y, con un bloque más corto o una
ventana más larga, contaminaría la fase que se usa. La regla práctica es elegir `fft_rows` dentro del bloque.

## 6. La fase de cada banda en la fila de evaluación

`fourier_column_phases` toma, en cada banda, la fase demodulada en las filas de la ventana, la **desenvuelve a lo
largo de $y$** y le ajusta un polinomio (una recta por defecto, `deg=1`); su valor en la fila de evaluación es la fase
de la banda y su pendiente da el periodo. Como se dispone de la fase en cada píxel, el polinomio puede ser de grado
mayor para seguir una fase curva a lo largo de $y$ (por ejemplo, `deg=2`). El resultado tiene la misma estructura
(`ColumnPhases`) que el ajuste sinusoidal de `interflib.fringe.edge_comparison`.

In [ ]:
x_platen = int(EDGE_X - 100)
d = demodulate_columns(img, [x_platen], FFT_ROWS)
sel = (y >= ROWS[0]) & (y < ROWS[1])
unwrapped = np.unwrap(d.phase[sel, 0])
coef = np.polyfit(y[sel] - EVAL_ROW, unwrapped, 1)
cp = fourier_column_phases(img, [x_platen], ROWS, EVAL_ROW, fft_rows=FFT_ROWS)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
axes[0].plot(y, d.phase[:, 0], ".", ms=2, color="tab:gray", label="fase envuelta")
axes[0].plot(y[sel], unwrapped - 2 * np.pi * np.floor(unwrapped[0] / (2 * np.pi)), color="tab:blue", lw=1.2,
             label="desenvuelta en la ventana")
axes[0].axvline(EVAL_ROW, color="tab:purple", ls=":")
axes[0].set_xlabel("fila y (px)"); axes[0].set_ylabel("fase (rad)"); axes[0].legend(fontsize=8)
resid = unwrapped - np.polyval(coef, y[sel] - EVAL_ROW)
axes[1].plot(y[sel], resid * 1e3, color="tab:blue", lw=1)
axes[1].set_xlabel("fila y (px)"); axes[1].set_ylabel("residuo de la recta (mrad)")
plt.tight_layout(); plt.show()
print(f"banda x = {x_platen}: fase en la fila {EVAL_ROW} = {cp.phase[0]:.4f} rad, "
      f"periodo = {cp.period[0]:.3f} px, R2 del modelo de franja = {cp.r2[0]:.5f}")

**Lectura de la figura.** A la izquierda, la fase envuelta de una banda de la platina a lo largo de las filas (puntos
grises, en dientes de sierra) y la misma fase desenvuelta en la ventana (azul), una rampa recta: su pendiente es la
portadora y su valor en la fila de evaluación (línea de puntos) es la fase de la banda. A la derecha, el residuo de la
recta ajustada, en milirradianes: menos de un milirradián, con una ondulación que crece hacia los extremos de la
ventana, la huella de la fuga espectral de los extremos del intervalo (sección 4). Si la fase tuviera curvatura a lo
largo de $y$, aparecería aquí como una parábola, que una recta promediaría en vez de seguir; de ahí la opción de un
polinomio de grado mayor. La salida impresa da la fase, el periodo y el $R^2$ con que el
modelo de franja reconstruido ($a+b\sin\varphi$) explica la intensidad de la banda, que sirve para descartar bandas
sin franjas.

## 7. La fracción en el borde

`fourier_edge_fraction` calcula la fase de las bandas a cada lado del borde, ajusta en cada lado un polinomio a lo
largo de $x$, los evalúa en la columna del borde y resta: en ese punto las dos fases comparten píxel, de modo que
todo término común a bloque y platina (la inclinación de la referencia, un pistón) se cancela, y la diferencia es
$2\pi F$. Esta comparación es la misma que usa el ajuste sinusoidal (`edge_comparison.fraction_at_edge`); el método
de Fourier solo cambia cómo se obtiene la fase de cada banda.

In [ ]:
res = fourier_edge_fraction(img, EDGE_X, ROWS, EVAL_ROW, fft_rows=FFT_ROWS)

fig, ax = plt.subplots(figsize=(9, 3.6))
for side, c, lab in [(res.platen, "tab:blue", "platina"), (res.block, "tab:orange", "bloque")]:
    ax.plot(side.x, side.phase_unwrapped / (2 * np.pi), ".", color=c, label=lab)
xx_p = np.array([EDGE_X - 180, EDGE_X]); xx_b = np.array([EDGE_X, EDGE_X + 180])
ax.plot(xx_p, (res.phase_platen + 2 * np.pi * res.slope_platen * (xx_p - EDGE_X)) / (2 * np.pi), color="tab:blue", lw=0.8)
ax.plot(xx_b, (res.phase_block + 2 * np.pi * res.slope_block * (xx_b - EDGE_X)) / (2 * np.pi), color="tab:orange", lw=0.8)
ax.axvline(EDGE_X, color="tab:red", lw=1)
ax.set_xlabel("columna x (px)"); ax.set_ylabel("fase en la fila de evaluacion (franjas)")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

gap = (res.fraction - scene.fraction_truth + 0.5) % 1.0 - 0.5
print(f"F recuperada = {res.fraction:.6f}")
print(f"F verdadera  = {scene.fraction_truth:.6f}")
print(f"brecha       = {gap:.1e} franjas ({gap * WAVELENGTH_NM / 2:.4f} nm en longitud, con lambda/2 por franja)")
print(f"gradiente comun en x: platina {res.slope_platen:.2e}, bloque {res.slope_block:.2e} franjas/px")

**Lectura de la figura.** La fase de cada banda en la fila de evaluación, en franjas, a cada lado del borde (línea
roja): platina en azul, bloque en naranja. Las dos nubes tienen la misma pendiente, que es la inclinación común de las
franjas; las rectas las prolongan hasta el borde, donde su diferencia de altura (módulo 1) es la fracción. La salida
impresa la compara con la verdad del simulador: coinciden a nivel de error numérico, y los dos gradientes en $x$
coinciden porque en esta escena no hay cuña entre bloque y platina.

## 8. Variante bidimensional

La versión por columnas filtra solo a lo largo de $y$. La variante bidimensional transforma en dos dimensiones un
**rectángulo que está entero sobre una superficie** (la platina junto al bloque, o la cara del bloque). En el espectro
2D, la franja es un lóbulo en $(g_x, g_y)$, donde $g_x$ es la componente que introduce la inclinación de las franjas;
el filtro es un disco de radio $\mathrm{bandwidth}\cdot|g|$ alrededor del lóbulo, de modo que limita también el
ancho de banda a lo largo de $x$ y entrega un mapa de fase de todo el rectángulo.

**Detección síncrona.** `demodulate_2d` no filtra el lóbulo donde está: primero multiplica el rectángulo por
$e^{-i2\pi(g_x x + g_y y)}$, que lleva el lóbulo a frecuencia cero, luego aplica un pasabajos y finalmente devuelve la
portadora [Servín *et al.* (2014), §4.2]. Las dos operaciones serían equivalentes si la imagen fuera periódica, pero
la transformada discreta trata el rectángulo como si se repitiera, y la fase de la portadora no vuelve a su valor al
cruzar de un extremo al opuesto. Filtrando el lóbulo en su sitio, ese salto se reparte por el filtro, que es estrecho
a lo largo de $x$, sobre las columnas cercanas a los extremos, justo donde está el borde del bloque. Tras quitar la
portadora, lo que queda varía despacio y casi no salta entre extremos.

**Una región por superficie, cada una con su portadora.** Una sola transformada sobre la imagen incluiría el escalón
del borde, que se repartiría por todo el espectro. Por eso `fourier_edge_fraction_2d` demodula por separado una región
de platina y una de bloque, y estima la portadora de cada una: si entre bloque y platina hay cuña, sus portadoras
difieren, y la detección síncrona necesita la de la superficie que demodula. Las regiones se extienden más allá de
las bandas que se usan (de `inner` px del borde a `span + outer` px), porque cerca de los extremos de una región la
fase se degrada; deben quedar, eso sí, enteras sobre su superficie.

In [ ]:
INNER, OUTER, SPAN, GAP = 12, 20, 180, 30
cols_platen = (max(0, int(np.floor(EDGE_X - SPAN - OUTER)) - 2), int(np.floor(EDGE_X - INNER)))
cols_block = (int(np.ceil(EDGE_X + INNER)), int(np.ceil(EDGE_X + SPAN + OUTER)) + 3)
region = img[FFT_ROWS[0]:FFT_ROWS[1], cols_platen[0]:cols_platen[1]]
g_platen = estimate_carrier_2d(region)
g_block = estimate_carrier_2d(img[FFT_ROWS[0]:FFT_ROWS[1], cols_block[0]:cols_block[1]])

ny, nx = region.shape
spec2d = np.fft.fftshift(np.abs(np.fft.fft2(region - region.mean())))
fy2, fx2 = np.fft.fftshift(np.fft.fftfreq(ny)), np.fft.fftshift(np.fft.fftfreq(nx))
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
show(img, title="regiones de la variante 2D", ax=axes[0])
for c, col in [(cols_platen, "tab:blue"), (cols_block, "tab:orange")]:
    axes[0].add_patch(plt.Rectangle((c[0], FFT_ROWS[0]), c[1] - c[0], FFT_ROWS[1] - FFT_ROWS[0],
                                    fill=False, ec=col, lw=1.5))
axes[0].axvline(EDGE_X, color="tab:red", lw=1)
axes[1].imshow(np.log10(spec2d + 1e-9), cmap="magma", aspect="equal",
               extent=[fx2[0], fx2[-1], fy2[-1], fy2[0]])
axes[1].plot(g_platen[0], g_platen[1], "o", mfc="none", mec="c", ms=14, mew=1.5)
axes[1].add_patch(plt.Circle(g_platen, 0.5 * np.hypot(*g_platen), fill=False, ec="c", ls="--"))
axes[1].set_xlim(-0.1, 0.1); axes[1].set_ylim(0.1, -0.1)
axes[1].set_xlabel(r"$f_x$ (franjas/px)"); axes[1].set_ylabel(r"$f_y$ (franjas/px)")
axes[1].set_title("espectro 2D de la region de platina (log)")
plt.tight_layout(); plt.show()
print(f"portadora platina (g_x, g_y) = ({g_platen[0]:.5f}, {g_platen[1]:.5f}) franjas/px")
print(f"portadora bloque  (g_x, g_y) = ({g_block[0]:.5f}, {g_block[1]:.5f}) franjas/px")

**Lectura de la figura.** A la izquierda, las dos regiones que demodula la variante 2D: la de platina (azul) a la
izquierda del borde y la de bloque (naranja) a la derecha, ambas entre las filas de la transformada y sin tocar el
borde. A la derecha, el espectro bidimensional de la región de platina en escala logarítmica: el fondo en el centro y
los dos lóbulos de la franja, casi sobre el eje vertical porque las franjas son casi horizontales; su pequeño
desplazamiento horizontal es $g_x$, la inclinación. Las rayas horizontales que atraviesan el fondo y los lóbulos son
la fuga espectral de los extremos de la región a lo largo de $x$: la transformada trata la región como periódica y la
franja no lo es. Es lo que la detección síncrona evita que llegue a la fase. El círculo discontinuo es el disco del
filtro, centrado en el lóbulo que estima `estimate_carrier_2d` (círculo pequeño). La salida impresa da la portadora de
cada región; en esta escena coinciden porque no hay cuña.

In [ ]:
maps = [demodulate_2d(img, FFT_ROWS, c) for c in (cols_platen, cols_block)]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
for m in maps:
    ext = [m.x[0], m.x[-1], m.y[-1], m.y[0]]
    axes[0].imshow(m.phase, cmap="twilight", extent=ext, aspect="auto")
    true = scene.phase[m.y[0]:m.y[-1] + 1, m.x[0]:m.x[-1] + 1]
    e = wrap_phase(m.phase - true)
    e = wrap_phase(e - np.angle(np.mean(np.exp(1j * e))))
    h = axes[1].imshow(e, cmap="RdBu_r", vmin=-0.01, vmax=0.01, extent=ext, aspect="auto")
for ax in axes:
    ax.set_xlim(0, img.shape[1]); ax.set_ylim(FFT_ROWS[1], FFT_ROWS[0])
    ax.axvline(EDGE_X, color="tab:red", lw=1); ax.set_xlabel("columna x (px)")
    ax.axvspan(EDGE_X - SPAN, EDGE_X - GAP, color="k", alpha=0.06)
    ax.axvspan(EDGE_X + GAP, EDGE_X + SPAN, color="k", alpha=0.06)
axes[0].set_title("fase envuelta de las dos regiones"); axes[0].set_ylabel("fila y (px)")
axes[1].set_title(r"$\varphi$ - verdad (rad)"); plt.colorbar(h, ax=axes[1], fraction=0.046, pad=0.04)
plt.tight_layout(); plt.show()

**Lectura de la figura.** A la izquierda, la fase envuelta de las dos regiones, colocadas en su posición en la imagen;
el corrimiento entre ellas en el borde (línea roja) es el que mide la fracción. A la derecha, la diferencia con la
fase verdadera, en la misma escala de $\pm 0{,}01$ rad que la sección 4: el error se concentra en los **bordes** de
cada región, sobre todo en las esquinas, y ya no solo arriba y abajo como en la versión por columnas, porque el filtro
actúa también a lo largo de $x$. Las zonas sombreadas son las columnas de las bandas que se usan (de 30 a 180 px del
borde): quedan dentro de las regiones, apartadas de sus extremos gracias a los márgenes `inner` y `outer`.

In [ ]:
res2d = fourier_edge_fraction_2d(img, EDGE_X, ROWS, EVAL_ROW, fft_rows=FFT_ROWS, inner=INNER, outer=OUTER)
gap2d = (res2d.fraction - scene.fraction_truth + 0.5) % 1.0 - 0.5
print(f"F (2D)           = {res2d.fraction:.6f}   brecha {gap2d:.1e} franjas")
print(f"F (por columnas) = {res.fraction:.6f}")
print(f"F verdadera      = {scene.fraction_truth:.6f}")

**Lectura.** La variante bidimensional entrega la misma clase de resultado (`EdgeFractionResult`) y, en esta escena,
recupera la fracción verdadera como la versión por columnas. Las dos comparten todo lo que sigue a la demodulación:
fase de cada banda en la fila de evaluación y comparación en el borde. Lo que añade la 2D es el mapa de fase de cada
región, que permite trabajar con la fase en dos dimensiones, a costa de exigir regiones que estén enteras sobre una
superficie y de un error mayor en los lados de cada región.

## 9. Parámetros

| Parámetro | Dónde | Qué controla | Criterio |
|---|---|---|---|
| `fft_rows` | todas | Filas de la transformada | Dentro de la región con franjas y sin cruzar un borde horizontal del bloque (sección 5); más ancho que `rows` |
| `rows`, `eval_row` | `fourier_column_phases`, `fourier_edge_fraction` | Filas cuya fase entra en el ajuste y fila a la que se refiere | Bien dentro de `fft_rows`, lejos de sus extremos (sección 4) |
| `carrier` | todas | Portadora $g_y$ (columnas) o $(g_x, g_y)$ (2D) | Si se omite, se estima: de las bandas de la platina (columnas, sección 3) o de cada región (2D, sección 8) |
| `bandwidth` | todas | Semiancho relativo del filtro, en $(0,1)$ | Debe dejar fuera el fondo; 0,5 por defecto |
| `min_cycles` | todas | Mínimo de franjas en el intervalo | Por debajo, la imagen se rechaza (`ValueError`) |
| `taper` | todas | Fracción del intervalo suavizada con una ventana de Tukey | Opcional; ablanda los extremos, que de todos modos quedan fuera de `rows` |
| `half_width` | todas | Columnas promediadas por banda ($2\,\text{half\_width}+1$) | 2 por defecto, como el ajuste sinusoidal |
| `deg_y` / `deg` | `fourier_edge_fraction` | Grado del polinomio a lo largo de $y$ (fase de cada banda) y a lo largo de $x$ (cada lado) | 1 por defecto; `deg_y=2` sigue una fase curva a lo largo de $y$ |
| `gap`, `span`, `step`, `r2_min` | `fourier_edge_fraction` | Bandas usadas a cada lado del borde y filtro de calidad | Como en el ajuste sinusoidal: de 30 a 180 px del borde, $R^2\ge 0{,}3$ |
| `inner`, `outer` | `fourier_edge_fraction_2d` | Extensión de las regiones 2D: de `inner` px del borde a `span + outer` px | Cada región entera sobre su superficie: `inner` fuera de la línea oscura del borde y menor que `gap`; `span + outer` antes del borde lejano de la platina o del bloque (sección 8) |

## 10. Cierre

La cadena recorrida:

$$\text{un interferograma con portadora}\;\xrightarrow{\;\text{espectro por columna, filtro}\;}\;
\varphi\ (\text{envuelta, cada píxel})\;\xrightarrow{\;\text{desenvolver en }y\text{, ajustar}\;}\;
\text{fase de cada banda}\;\xrightarrow{\;\text{extrapolar al borde}\;}\;F.$$

- La **portadora espacial** sustituye a los pasos de fase: una sola toma basta, sin mover la referencia.
- La transformada **por columnas** respeta la geometría bloque-platina: el borde vertical no contamina el espectro; los
  bordes horizontales se evitan con `fft_rows`.
- La **fase en cada píxel** permite ajustar a lo largo de $y$ un polinomio del grado que haga falta, y la
  **comparación en el borde** cancela todo término común a las dos superficies.
- La **variante bidimensional** demodula por detección síncrona una región de platina y otra de bloque y da un mapa de
  fase de cada una; termina en la misma comparación en el borde.

**Límites del método.** Necesita franjas suficientes para separar el lóbulo del fondo, y una escena que varíe despacio
frente a la franja; cerca de los extremos del intervalo y de los bordes del bloque la fase es menos fiable. La
fracción que entrega se refiere al borde del bloque en la fila de evaluación, no al centro de su cara.